In [ ]:
from deep_translator import GoogleTranslator
import random

### Data Perturbation by Dataset Fusion

Fusion with Friends-QIA

In [ ]:
with open("../../data/Experimental-Variants/friendsqia_train.tsv", "r", encoding="utf-8") as f:
    friends_data = [line.strip() for line in f.readlines()][1:]

with open(indirectqa_en_path, "r", encoding="utf-8") as f:
    data = [line.strip() for line in f.readlines()]
    header = data[0]
    data = data[1:]

indirect_friends = friends_data + data
random.shuffle(indirect_friends)
with open("../../data/Experimental-Variants/fusions/IndirectQA-en_plus_Friends-QIA.tsv", "w", encoding="utf-8") as f:
    f.write(header + "\n")
    for line in indirect_friends:
        f.write(line + "\n")

### Data Perturbation by Label Remapping and Omissions


REMAPPED

Old label set:
- 1: Yes
- 2: No
- 3: Conditional yes
- 4: Neither yes nor no
- 5: Other
- 6: Lacking context

New label set:
- 1: Yes (formerly Yes and Conditional Yes)
- 2: No
- 3: Neither yes nor no
- 4: Other (formerly Other and Lacking context)

In [ ]:
# remap labels to smaller label set to reduce learning effort and redundancies 
label_remapping = {1:1, 2:2, 3:1, 4:3, 5:4, 6:4}

def remap_labels(data_path, outfile_path):
    with open(data_path, "r", encoding="utf-8") as f:
        data = [line.strip().split("\t") for line in f.readlines()]
    
    remapped_data = [data[0]]

    for line in data[1:]:
        q, a, label = line
        remapped_label = str(label_remapping[int(label)])
        remapped_data.append([q, a, remapped_label])
    
    with open(outfile_path, "w", encoding="utf-8") as f:
        for line in remapped_data:
            f.write("\t".join(line) + "\n")

remap_labels("../../data/InQA_plus/EN/InQA_plus_en_all.tsv", "../../data/Experimental-Variants/remapped/IndirectQA_plus_en_remapped.tsv")
remap_labels("../../data/Experimental-Variants/fusions/IndirectQA-en-plus_Friends-QIA.tsv", "../../data/Experimental-Variants/remapped/IndirectQA-en_plus_Friends-QIA_remapped.tsv")

YESNO

New label set:
- 1: Yes
- 2: No

In [ ]:
# create dataset with only label 1 (yes) and 2 (no)
for language in ["EN", "DE", "BA"]:
    og_data_path = f"../../data/InQA_plus/{language}/InQA_plus_{language.lower()}_all.tsv"
    yesno_data_path = f"../../data/Experimental-Variants/yesno/IndirectQA_plus_{language.lower()}_yesno.tsv"

    with open(og_data_path, "r", encoding="utf-8") as f:
        data = [line.strip().split("\t") for line in f.readlines()]
        header = data[0]

    yes_pairs = [line for line in data if line[2]=="1"]
    no_pairs = [line for line in data if line[2]=="2"]

    print("Language:", language)
    print("Yes:", len(yes_pairs))
    print("No:", len(no_pairs))

    yesno = yes_pairs + no_pairs
    # random.shuffle(yesno)
    print("Yesno:", len(yesno), "\n")

    with open(yesno_data_path, "w", encoding="utf-8") as f:
        f.write("\t".join(header)+"\n")
        for line in yesno:
            f.write("\t".join(line)+"\n")

Language: EN
Yes: 183
No: 100
Yesno: 283 

Language: DE
Yes: 183
No: 100
Yesno: 283 

Language: BA
Yes: 183
No: 100
Yesno: 283 

